In [4]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
import numpy as np

In [5]:
torch.manual_seed(42)
data = load_wine()

In [9]:
print("Features:", data.feature_names)
print("Target classes:", data.target_names)

Features: ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']
Target classes: ['class_0' 'class_1' 'class_2']


In [10]:
print("Data shape:", data.data.shape)
print("Target shape:", data.target.shape)

Data shape: (178, 13)
Target shape: (178,)


In [11]:
print("First 5 samples:\n", data.data[:5])

First 5 samples:
 [[1.423e+01 1.710e+00 2.430e+00 1.560e+01 1.270e+02 2.800e+00 3.060e+00
  2.800e-01 2.290e+00 5.640e+00 1.040e+00 3.920e+00 1.065e+03]
 [1.320e+01 1.780e+00 2.140e+00 1.120e+01 1.000e+02 2.650e+00 2.760e+00
  2.600e-01 1.280e+00 4.380e+00 1.050e+00 3.400e+00 1.050e+03]
 [1.316e+01 2.360e+00 2.670e+00 1.860e+01 1.010e+02 2.800e+00 3.240e+00
  3.000e-01 2.810e+00 5.680e+00 1.030e+00 3.170e+00 1.185e+03]
 [1.437e+01 1.950e+00 2.500e+00 1.680e+01 1.130e+02 3.850e+00 3.490e+00
  2.400e-01 2.180e+00 7.800e+00 8.600e-01 3.450e+00 1.480e+03]
 [1.324e+01 2.590e+00 2.870e+00 2.100e+01 1.180e+02 2.800e+00 2.690e+00
  3.900e-01 1.820e+00 4.320e+00 1.040e+00 2.930e+00 7.350e+02]]


In [12]:
x_train, x_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42,stratify=data.target)

In [13]:
Sc = StandardScaler()
x_train = Sc.fit_transform(x_train)
x_test = Sc.transform(x_test)

In [14]:
x_tr = torch.tensor(x_train, dtype=torch.float32)
x_te = torch.tensor(x_test, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.long)
y_te = torch.tensor(y_test, dtype=torch.long)

In [15]:
print(x_tr.shape, x_tr.dtype)
print(x_te.shape, x_te.dtype)
print(y_tr.shape, y_tr.dtype)
print(y_te.shape, y_te.dtype)

torch.Size([142, 13]) torch.float32
torch.Size([36, 13]) torch.float32
torch.Size([142]) torch.int64
torch.Size([36]) torch.int64


In [18]:
train_ds = TensorDataset(x_tr, y_tr)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)

In [19]:
for xb, yb in train_dl:
    print(xb.shape, yb.shape)
    break

torch.Size([32, 13]) torch.Size([32])


In [24]:
class Net(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(16, 3)
        )
    def forward(self, x):
        return self.net(x)

In [25]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Net(x_tr.shape[1]).to(device)

In [26]:
print(model)

Net(
  (net): Sequential(
    (0): Linear(in_features=13, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=16, out_features=3, bias=True)
  )
)


In [27]:
print('device:', device)

device: cpu


In [28]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [30]:
for epoch in range (60):
    model.train()
    running = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running += loss.item()
    print(f"Epoch {epoch +1:3d}, Loss: {running/len(train_dl):.4f}")

Epoch   1, Loss: 1.1094
Epoch   2, Loss: 1.0921
Epoch   3, Loss: 1.0820
Epoch   4, Loss: 1.0482
Epoch   5, Loss: 1.0223
Epoch   6, Loss: 0.9940
Epoch   7, Loss: 0.9524
Epoch   8, Loss: 0.9481
Epoch   9, Loss: 0.9022
Epoch  10, Loss: 0.8537
Epoch  11, Loss: 0.7998
Epoch  12, Loss: 0.7736
Epoch  13, Loss: 0.7456
Epoch  14, Loss: 0.6864
Epoch  15, Loss: 0.6637
Epoch  16, Loss: 0.5972
Epoch  17, Loss: 0.5843
Epoch  18, Loss: 0.5667
Epoch  19, Loss: 0.5134
Epoch  20, Loss: 0.4767
Epoch  21, Loss: 0.4513
Epoch  22, Loss: 0.4125
Epoch  23, Loss: 0.4067
Epoch  24, Loss: 0.3340
Epoch  25, Loss: 0.3235
Epoch  26, Loss: 0.3170
Epoch  27, Loss: 0.2912
Epoch  28, Loss: 0.2717
Epoch  29, Loss: 0.2758
Epoch  30, Loss: 0.2649
Epoch  31, Loss: 0.2422
Epoch  32, Loss: 0.1998
Epoch  33, Loss: 0.2452
Epoch  34, Loss: 0.2119
Epoch  35, Loss: 0.1650
Epoch  36, Loss: 0.1627
Epoch  37, Loss: 0.1630
Epoch  38, Loss: 0.1766
Epoch  39, Loss: 0.1446
Epoch  40, Loss: 0.1588
Epoch  41, Loss: 0.1188
Epoch  42, Loss:

In [31]:
model.eval()
with torch.no_grad():
    logits = model(x_te.to(device))
    y_pred = torch.argmax(logits, dim=1)


In [32]:
accuracy = (y_pred == y_te).float().mean()
print(f"Test Accuracy: {accuracy:.2%}")

Test Accuracy: 97.22%


In [33]:
print("predicted_classes:", y_pred[:10].tolist())
print("true_classes:", y_te[:10].tolist())

predicted_classes: [0, 1, 0, 1, 1, 0, 0, 1, 1, 2]
true_classes: [0, 2, 0, 1, 1, 0, 0, 1, 1, 2]
